# 공공데이터포털 단기금융증권거래정보 API 데이터 수집
금융위원회 단기금융증권거래정보 - getCaseBuyAndSellInfo API

In [11]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os

In [12]:
# ============================================================
# 설정값
# ============================================================

# 공공데이터포털에서 발급받은 서비스키 (인코딩된 키 사용)
SERVICE_KEY = "b8f1d619e5e3d7e3901d31d1460de0ce9f0b25c0ccddb0cf26a19e4ad544ccfa"

# 조회 기간 설정 (YYYYMMDD 형식) - 테스트용 10일
START_DATE = "20240101"
END_DATE = "20241230"

# API 설정 - 오퍼레이션명(getCaseBuyAndSellInfo) 포함 필수
BASE_URL = "https://apis.data.go.kr/1160100/service/GetShorTermSecuTradInfoService/getCaseBuyAndSellInfo"
NUM_OF_ROWS = 1000  # 한 페이지당 요청 건수
RESULT_TYPE = "json"  # 응답 형식

# 재시도 설정
MAX_RETRIES = 3
RETRY_DELAY = 2  # 초
REQUEST_DELAY = 0.5  # API 호출 간 대기 시간 (초)

# 저장 파일명
OUTPUT_FILE = "short_term_securities_data_1.csv"

In [13]:
def generate_date_range(start_date: str, end_date: str) -> list:
    """
    시작일부터 종료일까지의 날짜 리스트 생성 (YYYYMMDD 형식)
    """
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    
    date_list = []
    current = start
    while current <= end:
        date_list.append(current.strftime("%Y%m%d"))
        current += timedelta(days=1)
    
    return date_list

In [14]:
def fetch_api_data(bas_dt: str, page_no: int = 1) -> dict:
    """
    API 호출 함수 (재시도 로직 포함)
    """
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": NUM_OF_ROWS,
        "pageNo": page_no,
        "resultType": RESULT_TYPE,
        "basDt": bas_dt
    }
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            return data
            
        except requests.exceptions.RequestException as e:
            print(f"  [오류] 시도 {attempt}/{MAX_RETRIES} - {bas_dt} 페이지 {page_no}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            else:
                print(f"  [실패] {bas_dt} 페이지 {page_no} 최대 재시도 횟수 초과")
                return None
        except Exception as e:
            print(f"  [오류] 예상치 못한 오류 - {bas_dt} 페이지 {page_no}: {e}")
            return None
    
    return None

In [15]:
def parse_items(data: dict) -> list:
    """
    API 응답에서 items 추출 (dict 또는 list 안전 처리)
    """
    try:
        # 응답 구조: response > body > items > item
        response = data.get("response", {})
        body = response.get("body", {})
        items = body.get("items", {})
        
        # items가 빈 문자열이거나 None인 경우
        if not items:
            return []
        
        # items에서 item 추출
        item = items.get("item", [])
        
        # item이 None인 경우
        if item is None:
            return []
        
        # item이 dict인 경우 (단일 항목) -> list로 변환
        if isinstance(item, dict):
            return [item]
        
        # item이 list인 경우
        if isinstance(item, list):
            return item
        
        return []
        
    except Exception as e:
        print(f"  [경고] 데이터 파싱 오류: {e}")
        return []

In [16]:
def get_total_count(data: dict) -> int:
    """
    전체 데이터 건수 추출
    """
    try:
        total_count = data.get("response", {}).get("body", {}).get("totalCount", 0)
        return int(total_count)
    except:
        return 0

In [17]:
def fetch_all_pages_for_date(bas_dt: str) -> list:
    """
    특정 날짜의 모든 페이지 데이터 수집
    """
    all_items = []
    page_no = 1
    
    # 첫 페이지 호출하여 전체 건수 확인
    first_data = fetch_api_data(bas_dt, page_no=1)
    
    if first_data is None:
        return []
    
    total_count = get_total_count(first_data)
    
    if total_count == 0:
        return []
    
    total_pages = (total_count + NUM_OF_ROWS - 1) // NUM_OF_ROWS
    print(f"  -> 총 {total_count}건, {total_pages}페이지")
    
    # 첫 페이지 데이터 추가
    items = parse_items(first_data)
    all_items.extend(items)
    
    # 나머지 페이지 순회
    for page_no in range(2, total_pages + 1):
        time.sleep(REQUEST_DELAY)
        
        data = fetch_api_data(bas_dt, page_no=page_no)
        
        if data is None:
            continue
        
        items = parse_items(data)
        all_items.extend(items)
        
        if page_no % 10 == 0:
            print(f"    페이지 {page_no}/{total_pages} 완료 (누적: {len(all_items)}건)")
    
    return all_items

In [18]:
def main():
    """
    메인 실행 함수
    """
    print("="*60)
    print("단기금융증권거래정보 API 데이터 수집 시작")
    print(f"조회 기간: {START_DATE} ~ {END_DATE}")
    print("="*60)
    
    # 날짜 범위 생성
    date_list = generate_date_range(START_DATE, END_DATE)
    total_dates = len(date_list)
    print(f"총 {total_dates}일 조회 예정\n")
    
    # 전체 데이터 저장용 리스트
    all_data = []
    
    # 날짜별 순회
    for idx, bas_dt in enumerate(date_list, 1):
        print(f"[{idx}/{total_dates}] {bas_dt} 처리 중...")
        
        items = fetch_all_pages_for_date(bas_dt)
        
        if items:
            all_data.extend(items)
            print(f"  -> {len(items)}건 수집 (총 누적: {len(all_data)}건)")
        else:
            print(f"  -> 데이터 없음")
        
        time.sleep(REQUEST_DELAY)
    
    print("\n" + "="*60)
    print("데이터 수집 완료")
    print(f"총 수집 건수: {len(all_data)}건")
    print("="*60)
    
    return all_data

In [19]:
# 데이터 수집 실행
all_data = main()

단기금융증권거래정보 API 데이터 수집 시작
조회 기간: 20240101 ~ 20241230
총 365일 조회 예정

[1/365] 20240101 처리 중...
  -> 데이터 없음
[2/365] 20240102 처리 중...
  -> 총 700건, 1페이지
  -> 700건 수집 (총 누적: 700건)
[3/365] 20240103 처리 중...
  -> 총 1165건, 2페이지
  -> 1165건 수집 (총 누적: 1865건)
[4/365] 20240104 처리 중...
  -> 총 1600건, 2페이지
  -> 1600건 수집 (총 누적: 3465건)
[5/365] 20240105 처리 중...
  -> 총 1485건, 2페이지
  -> 1485건 수집 (총 누적: 4950건)
[6/365] 20240106 처리 중...
  -> 데이터 없음
[7/365] 20240107 처리 중...
  -> 데이터 없음
[8/365] 20240108 처리 중...
  -> 총 1355건, 2페이지
  -> 1355건 수집 (총 누적: 6305건)
[9/365] 20240109 처리 중...
  -> 총 1034건, 2페이지
  -> 1034건 수집 (총 누적: 7339건)
[10/365] 20240110 처리 중...
  -> 총 660건, 1페이지
  -> 660건 수집 (총 누적: 7999건)
[11/365] 20240111 처리 중...
  -> 총 1226건, 2페이지
  -> 1226건 수집 (총 누적: 9225건)
[12/365] 20240112 처리 중...
  -> 총 1266건, 2페이지
  -> 1266건 수집 (총 누적: 10491건)
[13/365] 20240113 처리 중...
  -> 데이터 없음
[14/365] 20240114 처리 중...
  -> 데이터 없음
[15/365] 20240115 처리 중...
  -> 총 1498건, 2페이지
  -> 1498건 수집 (총 누적: 11989건)
[16/365] 20240116 처리 중...


In [20]:
# 컬럼명 한글 매핑
COLUMN_RENAME = {
    "basDt": "기준일자",
    "shtrFinPrdDcd": "단기금융상품구분코드",
    "shtrFinPrdDcdNm": "단기금융상품구분명",
    "isinCd": "ISIN코드",
    "isinCdNm": "ISIN코드명",
    "stlSqno": "결제일련번호",
    "curCd": "통화코드",
    "curCdNm": "통화명",
    "slngShtrFinBzcDcd": "매도업종코드",
    "slngShtrFinBzcDcdNm": "매도업종명",
    "buynShtrFinBzcDcd": "매수업종코드",
    "buynShtrFinBzcDcdNm": "매수업종명",
    "shtrPrdRmngExprDcd": "잔존만기코드",
    "shtrPrdRmngExprDcdNm": "잔존만기명",
    "shtrFinPrdTrdAmt": "거래금액",
    "shtrFinPrdIrt": "이자율",
    "shtrFinPrdIssuDt": "발행일자",
    "shtrFinPrdExprDt": "만기일자"
}

# DataFrame 생성 및 후처리
if all_data:
    df = pd.DataFrame(all_data)
    
    print(f"원본 데이터 건수: {len(df)}건")
    
    # 중복 제거
    df_deduplicated = df.drop_duplicates()
    removed_count = len(df) - len(df_deduplicated)
    print(f"중복 제거: {removed_count}건 제거됨")
    print(f"최종 데이터 건수: {len(df_deduplicated)}건")
    
    # 컬럼명 한글로 변경
    df_deduplicated = df_deduplicated.rename(columns=COLUMN_RENAME)
    print("\n컬럼명을 한글로 변경했습니다.")
    
    # 데이터 미리보기
    print("\n[데이터 미리보기]")
    display(df_deduplicated.head(10))
    
    print("\n[컬럼 정보]")
    print(df_deduplicated.columns.tolist())
else:
    print("수집된 데이터가 없습니다.")
    df_deduplicated = pd.DataFrame()

원본 데이터 건수: 302388건
중복 제거: 0건 제거됨
최종 데이터 건수: 302388건

컬럼명을 한글로 변경했습니다.

[데이터 미리보기]


,기준일자,단기금융상품구분코드,ISIN코드,결제일련번호,매도업종코드,단기금융상품구분명,통화코드,매도업종명,통화명,매수업종코드,매수업종명,ISIN코드명,잔존만기코드,잔존만기명,거래금액,이자율,발행일자,만기일자
0,20240102,14,KRZS00127G15,10000,0101,단기사채,KRW,증권사,KRW,0204,집합투자,부국증권 20240102-9-1(단),3,7일이상 1월미만,4995068494,4,20240102,20240111
1,20240102,14,KRZS47171002,1000000,0101,단기사채,KRW,증권사,KRW,0101,증권사,모니모송도베스트제일차 20231211-30-1(단),3,7일이상 1월미만,3996313425,4.205,20231211,20240110
2,20240102,14,KRZS47171002,1001000,0201,단기사채,KRW,증권사(신탁),KRW,0101,증권사,모니모송도베스트제일차 20231211-30-1(단),3,7일이상 1월미만,3996309042,4.21,20231211,20240110
3,20240102,14,KRZS008679L1,1004000,0201,단기사채,KRW,증권사(신탁),KRW,0101,증권사,신한투자증권 20231218-91-42(단),4,1월이상 3월미만,99156713,4.05,20231218,20240318
4,20240102,12,KRZF14017JDQ,101000,0101,CP,KRW,증권사,KRW,0101,증권사,KB국민카드 20231222-171-51,5,3월이상 1년미만,4910136987,4.1,20231222,20240610
5,20240102,14,KRZS46498008,1013000,0101,단기사채,KRW,증권사,KRW,0101,증권사,드래곤필드제일차 20240102-29-1(단),3,7일이상 1월미만,16338758357,4.7,20240102,20240131
6,20240102,12,KRZFA6732HD9,1019000,0101,CP,KRW,증권사,KRW,0204,집합투자,위드지엠제이십일차 20230918-154-17,4,1월이상 3월미만,4972383562,4.2,20230918,20240219
7,20240102,12,KRZFA6732KD9,1020000,0101,CP,KRW,증권사,KRW,0204,집합투자,위드지엠제이십일차 20230918-154-19,4,1월이상 3월미만,4972383562,4.2,20230918,20240219
8,20240102,12,KRZFA6732LD9,1021000,0101,CP,KRW,증권사,KRW,0204,집합투자,위드지엠제이십일차 20230918-154-20,4,1월이상 3월미만,4972383562,4.2,20230918,20240219
9,20240102,12,KRZFA6732JD9,1022000,0101,CP,KRW,증권사,KRW,0204,집합투자,위드지엠제이십일차 20230918-154-18,4,1월이상 3월미만,4972383562,4.2,20230918,20240219



[컬럼 정보]
['기준일자', '단기금융상품구분코드', 'ISIN코드', '결제일련번호', '매도업종코드', '단기금융상품구분명', '통화코드', '매도업종명', '통화명', '매수업종코드', '매수업종명', 'ISIN코드명', '잔존만기코드', '잔존만기명', '거래금액', '이자율', '발행일자', '만기일자']


In [21]:
# CSV 파일 저장 (UTF-8-sig: Excel에서 한글 깨짐 방지)
if not df_deduplicated.empty:
    df_deduplicated.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    
    file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)  # MB
    print(f"\n파일 저장 완료!")
    print(f"파일명: {OUTPUT_FILE}")
    print(f"파일 크기: {file_size:.2f} MB")
    print(f"저장 건수: {len(df_deduplicated)}건")
else:
    print("저장할 데이터가 없습니다.")


파일 저장 완료!
파일명: short_term_securities_data_1.csv
파일 크기: 49.94 MB
저장 건수: 302388건


In [22]:
# 데이터 기본 통계 확인 (선택사항)
if not df_deduplicated.empty:
    print("[기본 통계]")
    print(f"날짜 범위: {df_deduplicated['기준일자'].min()} ~ {df_deduplicated['기준일자'].max()}")
    print(f"고유 날짜 수: {df_deduplicated['기준일자'].nunique()}일")
    print("\n[날짜별 데이터 건수]")
    print(df_deduplicated['기준일자'].value_counts().sort_index().head(20))

[기본 통계]
날짜 범위: 20240102 ~ 20241230
고유 날짜 수: 244일

[날짜별 데이터 건수]
기준일자
20240102     700
20240103    1165
20240104    1600
20240105    1485
20240108    1355
20240109    1034
20240110     660
20240111    1226
20240112    1266
20240115    1498
20240116    1073
20240117    1157
20240118     936
20240119    1040
20240122    1205
20240123     881
20240124     793
20240125    1218
20240126     957
20240129    1232
Name: count, dtype: int64


In [23]:
print(f"END_DATE = '{END_DATE}'")
print(f"길이: {len(END_DATE)}")

END_DATE = '20241230'
길이: 8
